## Feature generation

In [7]:
import pandas as pd
import numpy as np
import skfp.fingerprints as sfp
from skfp.fingerprints import PhysiochemicalPropertiesFingerprint
from modules.data_loader import *

In [8]:
visco = load_cleaned()

In [9]:
visco.info()

<class 'pandas.DataFrame'>
RangeIndex: 8152 entries, 0 to 8151
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8152 non-null   str    
 1   type_surfactant1    8152 non-null   str    
 2   smiles_surfactant1  8152 non-null   str    
 3   c_surfactant1       8152 non-null   float64
 4   id_surfactant2      2726 non-null   str    
 5   type_surfactant2    2726 non-null   str    
 6   smiles_surfactant2  2631 non-null   str    
 7   c_surfactant2       8152 non-null   float64
 8   id_additive1        5504 non-null   str    
 9   smiles_additive1    5504 non-null   str    
 10  c_additive1         8152 non-null   float64
 11  id_additive2        1295 non-null   str    
 12  smiles_additive2    1139 non-null   str    
 13  c_additive2         8152 non-null   float64
 14  temperature         8152 non-null   int64  
 15  viscoelasticity     8152 non-null   float64
 16  g_0              

In [10]:
visco.head()

,id_surfactant1,type_surfactant1,smiles_surfactant1,c_surfactant1,id_surfactant2,type_surfactant2,smiles_surfactant2,c_surfactant2,id_additive1,smiles_additive1,...,id_additive2,smiles_additive2,c_additive2,temperature,viscoelasticity,g_0,g_min,omega_cr,tau_r,eta_0
0,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,NaN,NaN,0.0,25,0.0,NaN,NaN,NaN,NaN,0.02
1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,NaN,NaN,0.0,25,1.0,4.0,2.70,150.0,0.01,0.03
2,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,NaN,NaN,0.0,25,1.0,4.5,1.20,10.0,0.10,0.45
3,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,NaN,NaN,0.0,25,1.0,5.0,0.50,0.8,1.25,6.25
4,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,NaN,NaN,0.0,25,1.0,4.2,0.35,1.5,0.67,2.80


In [11]:
smiles = visco['smiles_surfactant1']
smiles = pd.concat([smiles, visco['smiles_surfactant2']])
smiles = pd.concat([smiles, visco['smiles_additive1']])
smiles = pd.concat([smiles, visco['smiles_additive2']])
smiles = smiles.drop_duplicates()
smiles = smiles.dropna()
len(smiles)

137

In [12]:
fp = PhysiochemicalPropertiesFingerprint(fp_size=256)
surfactant_fingerprints = fp.transform(smiles.values)

In [13]:
fingerprint_lookup = {
    smiles: fp for smiles, fp in zip(smiles, surfactant_fingerprints)
}

In [14]:
physio = pd.DataFrame(smiles, columns=['smiles'])

surfactant_fp = np.stack(
    physio["smiles"].map(fingerprint_lookup).to_numpy()
)

fp_df = pd.DataFrame(
    surfactant_fp,
    index=physio.index,
    columns=[f"fp_{i}" for i in range(256)]
)

physio = pd.concat([physio, fp_df], axis=1)

In [15]:
physio.info()

<class 'pandas.DataFrame'>
Index: 137 entries, 0 to 4640
Columns: 257 entries, smiles to fp_255
dtypes: str(1), uint8(256)
memory usage: 36.4 KB


In [16]:
save_features(physio, "physiochemical")